# local_MPC

Run and cache compare-ready Local MPC rollouts for both perfect and LSTM forecasts so `compare.ipynb` can replay them without re-solving.

Local MPC uses an `economic_only` objective. If you need SOC soft constraints, use the global MPC or ADMM MPC notebooks instead.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "configs").exists():
    repo_root = repo_root.parent
if not (repo_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import configs as configs_pkg
from configs.profiles import compose_experiment_config
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts.utils import local_mpc_rollout_packages as local_mpc_nb
from scripts.utils.project_paths import project_root as resolve_project_root

configs_pkg = importlib.reload(configs_pkg)
grid_nb = importlib.reload(grid_nb)
local_mpc_nb = importlib.reload(local_mpc_nb)


In [ ]:
PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"

TEST_START_DATE = "2020-06-01"
TEST_END_DATE = "2020-06-04"
BASE_PREDICTION_MODE = "normal"
LOAD_SCALE = [10.0] * 5
PV_SCALE = [5.0] * 5
BATTERY_CAPACITY_KWH = 20.0
BATTERY_MAX_POWER_KW = 10.0
BATTERY_MAX_CHARGE_RATE = BATTERY_MAX_POWER_KW / BATTERY_CAPACITY_KWH
BATTERY_CONTROLS = {"battery_capacity": BATTERY_CAPACITY_KWH, "max_charge_rate": BATTERY_MAX_CHARGE_RATE}
SAVE_LOCAL_MPC_ROLLOUT = True
LOCAL_MPC_PERFECT_TAG_OVERRIDE = None
LOCAL_MPC_LSTM_TAG_OVERRIDE = None


In [ ]:
base_cfg = compose_experiment_config(
    profile="base",
    algorithm="MATD3",
    model_family="mlp",
    data_dir=DATA_DIR,
    runtime_mode="performance",
)
grid_nb.apply_notebook_experiment_settings(
    base_cfg,
    prediction_mode=BASE_PREDICTION_MODE,
    test_start_date=TEST_START_DATE,
    test_end_date=TEST_END_DATE,
    load_scale=LOAD_SCALE,
    pv_scale=PV_SCALE,
    battery_controls=BATTERY_CONTROLS,
)
normal_comparison_cfg = grid_nb.build_comparison_cfg(base_cfg, prediction_mode="normal")
normal_comparison_cfg.runtime.forecast_ready = grid_nb.ensure_forecast_ready(normal_comparison_cfg)

display(
    pd.Series(
        {
            "test_start_date": base_cfg.data.test_start_date,
            "test_end_date": base_cfg.data.test_end_date,
            "base_prediction_mode": BASE_PREDICTION_MODE,
            "normal_forecast_backend": normal_comparison_cfg.forecast.type,
            "future_horizon": base_cfg.env.future_horizon,
            "episode_limit": base_cfg.env.episode_limit,
            "local_mpc_objective_mode": "economic_only",
            "import_price_markup_eur_per_kwh": float(base_cfg.reward.import_price_markup_eur_per_kwh),
            "export_subsidy_eur_per_kwh": float(base_cfg.reward.export_subsidy_eur_per_kwh),
            "normal_forecast_ready": normal_comparison_cfg.runtime.forecast_ready is not None,
        },
        name="local_mpc_notebook_config",
    )
)


In [ ]:
print(f"Starting {local_mpc_nb.LOCAL_MPC_PERFECT_LABEL}...")
local_mpc_perfect = grid_nb.collect_local_mpc_rollout(
    base_cfg,
    prediction_mode="perfect",
    label=local_mpc_nb.LOCAL_MPC_PERFECT_LABEL,
)
print("Done:", local_mpc_perfect.meta.get("controller", local_mpc_nb.LOCAL_MPC_PERFECT_LABEL))

print(f"Starting {local_mpc_nb.LOCAL_MPC_LSTM_LABEL}...")
local_mpc_lstm = grid_nb.collect_local_mpc_rollout(
    base_cfg,
    prediction_mode="normal",
    label=local_mpc_nb.LOCAL_MPC_LSTM_LABEL,
)
print("Done:", local_mpc_lstm.meta.get("controller", local_mpc_nb.LOCAL_MPC_LSTM_LABEL))


In [ ]:
metrics_df = grid_nb.compare_rollout_metrics(local_mpc_perfect, local_mpc_lstm)
display(metrics_df)


def _build_local_mpc_rollout_tag(cfg, *, prediction_mode: str, override: str | None = None):
    comparison_cfg = grid_nb.build_comparison_cfg(cfg, prediction_mode=prediction_mode)
    tag = override or (
        f"{comparison_cfg.data.test_start_date}_{comparison_cfg.data.test_end_date}_"
        f"agents{comparison_cfg.env.num_agents}_{prediction_mode}_{comparison_cfg.forecast.type}"
    )
    return tag, comparison_cfg


def _build_local_mpc_diagnostics(rollout):
    return {
        "controller": rollout.meta.get("controller"),
        "forecast_backend": rollout.meta.get("forecast_backend"),
        "local_mpc_solve_count": int(rollout.meta.get("local_mpc_solve_count", 0)),
        "local_mpc_solver_build_count": int(rollout.meta.get("local_mpc_solver_build_count", 0)),
        "local_mpc_solver_reuse_count": int(rollout.meta.get("local_mpc_solver_reuse_count", 0)),
        "local_mpc_total_solve_time_sec": float(rollout.meta.get("local_mpc_total_solve_time_sec", 0.0)),
        "local_mpc_avg_solve_time_sec": float(rollout.meta.get("local_mpc_avg_solve_time_sec", 0.0)),
    }


local_mpc_perfect_package_dir = None
local_mpc_lstm_package_dir = None
if SAVE_LOCAL_MPC_ROLLOUT:
    local_mpc_perfect_tag, local_mpc_perfect_cfg = _build_local_mpc_rollout_tag(
        base_cfg,
        prediction_mode="perfect",
        override=LOCAL_MPC_PERFECT_TAG_OVERRIDE,
    )
    local_mpc_lstm_tag, local_mpc_lstm_cfg = _build_local_mpc_rollout_tag(
        base_cfg,
        prediction_mode="normal",
        override=LOCAL_MPC_LSTM_TAG_OVERRIDE,
    )
    local_mpc_perfect_package = local_mpc_nb.build_local_mpc_rollout_package(
        local_mpc_perfect,
        controller_label=local_mpc_perfect.meta["controller"],
        cfg=base_cfg,
        prediction_mode="perfect",
        extra_meta={"rollout_meta": dict(local_mpc_perfect.meta)},
        diagnostic_summary=_build_local_mpc_diagnostics(local_mpc_perfect),
    )
    local_mpc_lstm_package = local_mpc_nb.build_local_mpc_rollout_package(
        local_mpc_lstm,
        controller_label=local_mpc_lstm.meta["controller"],
        cfg=base_cfg,
        prediction_mode="normal",
        extra_meta={"rollout_meta": dict(local_mpc_lstm.meta)},
        diagnostic_summary=_build_local_mpc_diagnostics(local_mpc_lstm),
    )
    local_mpc_perfect_package_dir = local_mpc_nb.save_local_mpc_rollout_package(
        local_mpc_perfect_package,
        PROJECT_ROOT / "artifacts" / "local_mpc_cached_rollout" / local_mpc_perfect_tag,
    )
    local_mpc_lstm_package_dir = local_mpc_nb.save_local_mpc_rollout_package(
        local_mpc_lstm_package,
        PROJECT_ROOT / "artifacts" / "local_mpc_cached_rollout" / local_mpc_lstm_tag,
    )

display(
    pd.Series(
        {
            "save_local_mpc_rollout": bool(SAVE_LOCAL_MPC_ROLLOUT),
            "local_mpc_perfect_controller": local_mpc_perfect.meta.get("controller"),
            "local_mpc_lstm_controller": local_mpc_lstm.meta.get("controller"),
            "local_mpc_perfect_package_dir": None if local_mpc_perfect_package_dir is None else str(local_mpc_perfect_package_dir),
            "local_mpc_lstm_package_dir": None if local_mpc_lstm_package_dir is None else str(local_mpc_lstm_package_dir),
        },
        name="local_mpc_rollout_cache",
    )
)


In [ ]:
grid_nb.plot_price_prediction_comparison(local_mpc_perfect, local_mpc_lstm)
plt.show()

grid_nb.plot_net_load_comparison(local_mpc_perfect, local_mpc_lstm)
plt.show()

grid_nb.plot_battery_power_and_soc_comparison(local_mpc_perfect, local_mpc_lstm)
plt.show()
